# MNIST

**Objetivo:** entrenar una red neuronal simple, que reconozca dígitos escritos a mano del dataset (MNIST)

**Framework utilizado: Tensorflow / Keras**. La elección se debe a que su API Sequential permite estructurar redes densas de forma rápida y directa. Ideal para comparar activaciones y optimizadores en pocas líneas de código.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix

# SEED -> Garantiza la reproducibilidad de los resultados
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 1. Carga y exploración de los datos

MNINST trae 70.000 imágenes de 28x28 píxeles en escalses de grises. Uso (60.000) para entrenar, 10.000 para test). Cada imagen tiene una etiqueta del 0 al 9

In [ ]:
# Descarga MNIST desde Keras
(X_train_img, y_train), (X_test_img, y_test) = keras.datasets.mnist.load_data()

# Mostramos cuántas imágenes hay y de que tamaño son
print("Train: ", X_train_img.shape)
print("Test: ", X_test_img.shape)

# Muestra 10 ejemplos
n_mostrar = 10
plt.figure(figsize=(10, 2))
for i in range(n_mostrar):
    plt.subplot(1, n_mostrar, i + 1)
    plt.imshow(X_train_img[i], cmap="gray")
    plt.title(str(y_train[i]))
    plt.axis("off")

plt.show()

## Preprocesamiento: normalizar y aplanar

- **Normalizar:** los píxeles van de 0 a 255. Se divide por 255 para dejarlos en 0 y 1, así la red aprende más rápido.
- **Aplanar:** la red densa no entiende imágenes 2D, así que cada imagen de 28x28 la convertimos en un vector de 784 números.

In [ ]:
INPUT_DIM = 28 * 28
X_train = (X_train_img.astype("float32") / 255.0).reshape(-1, INPUT_DIM)
X_test = (X_test_img.astype("float32") / 255).reshape(-1, INPUT_DIM)

print(f"Train aplanado: {X_train.shape}")
print(f"Valor mínimo: {X_train.min()} | máximo: {X_train.max()}")

## 3. Arquitectura: el cerebro

Red con **1 sola capa oculta**
- **Entrada:** 784 neuronas (una por píxel)
- **Capa Oculta:** 128 neuronas con activación ReLU o Sigmoid (para comparar)
- **Salida:** 10 neuronas con softmax (una probabilidad por cada dígito 0-9)

In [ ]:
# Función que arma la misma red cambiando solo activación y optimizador
def crear_modelo(activacion, optimizador):
    # LR igual para comparar de forma justa
    lr = 0.001
    if optimizador == "adam":
        opt = keras.optimizers.Adam(learning_rate=lr)
    else:
        opt = keras.optimizers.SGD(learning_rate=lr)

    modelo = keras.Sequential(
        [
            keras.layers.Input(shape=(784,), name="entrada"),
            keras.layers.Dense(128, activation=activacion, name="oculta"),
            keras.layers.Dense(10, activation="softmax", name="salida"),
        ]
    )

    modelo.compile(
        optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"]
    )

    return modelo


# Mostrar el resumen con ReLU + Adam como ejemplo
modelo_demo = crear_modelo(activacion="relu", optimizador="adam")
modelo_demo.summary()

## 4. Experimentos: activaciones vs optimizadores

**Mismo hiperparámetros base para todo:** 5 épocas, batch 128, lr 0.001, 10% de train como validación.
- Experimento 1 (activación): `ReLU + Adam` vs `Sigmoid + Adam`
- Experimento 2 (optimizador): el ganador anterior con `Adam` vs con `SGD`

In [ ]:
epochs = 5
batch_size = 128

# Experimento de activaciones (optimizador fijo: Adam)
modelo_relu_adam = crear_modelo(activacion="relu", optimizador="adam")

hist_relu_adam = modelo_relu_adam.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose="auto",
)

modelo_sigmoid_adam = crear_modelo(activacion="sigmoid", optimizador="adam")

hist_sigmoid_adam = modelo_sigmoid_adam.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose="auto",
)

# Experimento de optimizadores (activacion fija: ReLU)
modelo_relu_sgd = crear_modelo(activacion="relu", optimizador="sgd")

hist_relu_sgd = modelo_relu_sgd.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose="auto",
)

### Gráficas de los experimentos

In [ ]:
plt.figure(figsize=(12, 4))

# Curva de pérdida
plt.subplot(1, 2, 1)
plt.plot(hist_relu_adam.history["loss"], label="ReLU+Adam train")
plt.plot(hist_relu_adam.history["val_loss"], linestyle="--", label="Relu+Adam val")
plt.plot(hist_sigmoid_adam.history["loss"], label="Sigmoid+Adam train")
plt.plot(
    hist_sigmoid_adam.history["val_loss"], linestyle="--", label="Sigmoid+Adam val"
)
plt.plot(hist_relu_sgd.history["loss"], label="ReLU+SGD train")
plt.plot(hist_relu_sgd.history["val_loss"], linestyle="--", label="ReLU+SGD val")
plt.title("Loss por epoch (más bajo = mejor)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Curva de accuracy
plt.subplot(1, 2, 2)
plt.plot(hist_relu_adam.history["val_accuracy"], label="ReLU+Adam val")
plt.plot(hist_sigmoid_adam.history["val_accuracy"], label="Sigmoid+Adam val")
plt.plot(hist_relu_sgd.history["val_accuracy"], label="ReLU+SGD val")
plt.title("Accuracy de validación por epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Resumen numérico del último epoch para comparar fácil
print("Val accuracy final Relu+Adam: ", hist_relu_adam.history["val_accuracy"][-1])
print(
    "Val accuracy final Sigmoid+Adam: ", hist_sigmoid_adam.history["val_accuracy"][-1]
)
print("Val accuracy final Relu+SGD: ", hist_relu_sgd.history["val_accuracy"][-1])

**Observaciones del gráfico.**

- 1. 🔵**Relu + Adam.**

    Ambas líneas de pérdida (Loss) caen en picada de forma casi idéntica y se quedan pegadas al suelo desde la primera época.     
    Su precisión (Accuracy) en arranca muy alta y llega al ~96%.
    
    *Es el alumno estrella. Aprendió todo lo que necesitaba casi de inmediato. Como las líneas van de la mano en el piso, sabemos que el modelo es sumamente estable y no tiene ningún rastro de Overfitting.*

- 2. 🟢**Sigmoid + Adam.** 

    Al igual que el anterior, las dos curvas bajan juntas en sintonía. Arrancan un poquito más arriba en pérdida que `ReLU`, pero para la época 4 ya están casi en el mismo nivel y logran un gran Accuracy (~95%)

    *Es el alumno que se esfuerza. Le cuesta un poquito más arrancar debido a la función Sigmoid, pero gracias al optimizador `Adam`, logra entender el patrón rápido, de manera limpia y sin errores de memorización.*

- 3. 🟣**ReLU + SGD.**

    Las líneas van juntas, pero bajan de forma dolorosamente lenta. Su precisión en la gráfica derecha apenas va subiendo como una rampa recta y se queda abajo en ~84%.

    *Es el alumno lento. El camino que toma `(SGD)` da pasos tan cortos que 5 épocas no le alcanzan para terminar de procesar los datos. No está mal configurado y no tiene overfitting, simplemente le falta mucho más tiempo (épocas) para llegar a la meta.*






## 5. Evaluación: pruebas de campo

Se elige como ganador **Relu + Adam**. 
Lo evaluamos en test, que la red nunca vió.



In [ ]:
# El modelo ganador es ReLU + Adam
modelo_ganador = modelo_relu_adam

# Accuracy final en test 
test_loss, test_acc = modelo_ganador.evaluate(X_test, y_test, verbose="auto")
print("Accuracy en test: ", test_acc)

# Predecimos todos los dígitos del test
y_proba = modelo_ganador.predict(X_test, verbose="auto")
y_pred = np.argmax(y_proba, axis=1) # nos quedamos con el dígito más probable

# Matriz de confusión: filas=real, columnas=predicción. La diagonal es lo correcto.
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
plt.matshow(cm, cmap="Blues")
plt.title("Matriz de confusión (test)")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.colorbar()
plt.show()

In [ ]:
# Busca un acierto y un error para mostrar.
aciertos = np.where(y_pred == y_test)[0] # índices donde acertó
errores = np.where(y_pred != y_test)[0] # índices donde falló

idx_acierto = aciertos[0]
idx_error = errores[0]

plt.figure(figsize=(8,3))

# Ejemplo bien clasificado
plt.subplot(1, 2, 1)
plt.imshow(X_test_img[idx_acierto], cmap="gray")
plt.title(f"Acierto: real={y_test[idx_acierto]}, pred={y_pred[idx_acierto]}")
plt.axis("off")

# Ejemplo mal clasificado
plt.subplot(1, 2, 2)
plt.imshow(X_test_img[idx_error], cmap="gray")
plt.title(f"Error: real={y_test[idx_error]}, pred={y_pred[idx_error]}")
plt.axis("off")
plt.show()

print("Índice del error mostrado", idx_error)

**Análisis del error**

Ejemplo concreto: en test el modelo llega a ~97% pero falla unas ~300veces. Un caso típico es confundir un `4` con un `9` (o un `3` con un `8`) cuando el trazo está inclincado o mal cerrado.

Por qué creo que pasó: esta red tiene 1 sola capa densa de 128 neuronas, entonces no ve formas, ve pixeles aplanados (784 números). Si dos dígitos comparten muchos píxeles prendidos en el centro, la red los ve casi iguales y eleige el vecino más parecido por probabilidad. Una CNN miraría bordes y curvas locales, y rendiría mejor.